# 01. TPU v6e (Trillium) Roofline & TorchTPU 4 Execution Modes + XProf

> **Reference Docs**: [TorchTPU Execution Model](https://torchtpu.readthedocs.io/en/0.2.0/torchtpu_execution_model/) · [Performance Profiling with XProf](https://torchtpu.readthedocs.io/en/0.2.0/performance_profiling_with_xprof/)

이 노트북에서는 **Cloud TPU v6e (`ct6e-standard-1t`)** 위에서 다음 세 가지를 실습합니다:
1. `torch.tpu.get_device_properties()`로 **TPU v6e (Trillium) 하드웨어 스펙**(~918 TFLOPS `bf16`, ~1,638 GB/s HBM, Ridge Point ~560 FLOPs/Byte)을 조회하고 **Roofline 차트** 베이스라인을 그립니다.
2. 동일한 Transformer MLP+RMSNorm 블록을 **TorchTPU 4대 실행 모드**(`Strict Eager`, `Debug Eager`, `DeferAndFuse`, `torch.compile(backend="tpu")`)로 실행해 처리량(TFLOPS)과 Roofline 상의 위치 변화를 비교합니다.
3. `torch.profiler` + `TpuProfilerConfig`로 `.xplane.pb` 트레이스를 캡처하고 `XProf` / TensorBoard에서 **Microbubbles**와 **Operator Fusion**을 직접 진단합니다.

## 0. 환경 점검 및 설치

- **Cloud TPU v6e VM (`scripts/setup_v6e_vm.sh` 완료 환경)**: 이미 설치되어 있으므로 바로 통과합니다.
- **신규 Colab 런타임**: 자동 설치 후 **`런타임 → 세션 다시 시작`**을 한 번 눌러주세요.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch_tpu") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "setuptools<81"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch_tpu", "xprof", "tensorboard", "tensorboard-plugin-profile", "matplotlib",
    ])
    print("설치 완료! Colab 상단 [런타임 -> 세션 다시 시작] 후 다음 셀부터 실행하세요.")
else:
    print("TorchTPU 환경이 이미 준비되어 있습니다.")

## 1. TPU v6e 초기화 & Roofline 베이스라인 구축

In [ ]:
import os
import pathlib
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 4칩 VM(ct6e-standard-4t)에서 단일 칩 실습 시 0번 칩만 사용하도록 기본 지정
os.environ.setdefault("TPU_VISIBLE_DEVICES", "0")
# Tier-2 로컬 공유메모리(/dev/shm) 컴파일 캐시 설정
os.makedirs("/dev/shm/torch_tpu_cache/sprint_cache", exist_ok=True)
os.environ.setdefault("TORCH_TPU_TIER2_COMPILATION_CACHE", "sprint_cache")

device = torch.device("tpu")
props = torch.tpu.get_device_properties(device)

# TPU 세대별 HBM 대역폭 자동 매핑 (기본 타겟: TPU v6e Trillium = 1,638 GB/s, 918 TFLOPS)
HBM_BW_MAP = {"v6e": 1638.0, "v5p": 2765.0, "v5e": 819.0}
PEAK_HBM_BW_GBS = next((bw for key, bw in HBM_BW_MAP.items() if key in props.name.lower()), 1638.0)
PEAK_TFLOPS_BF16 = getattr(props, "bf16_flops_per_device", 918e12) / 1e12
HBM_GB = getattr(props, "hbm_bytes_per_device", 32 * 1024**3) / (1024**3)
RIDGE_POINT = (PEAK_TFLOPS_BF16 * 1e12) / (PEAK_HBM_BW_GBS * 1e9)

print(f"Device            : {props.name} ({device})")
print(f"PyTorch Version   : {torch.__version__}")
print(f"Default EagerMode : {torch.tpu.eager_mode}")
print(f"HBM Capacity      : {HBM_GB:.2f} GiB")
print(f"Peak BF16 Compute : {PEAK_TFLOPS_BF16:.1f} TFLOPS")
print(f"Peak HBM Bandwidth: {PEAK_HBM_BW_GBS:.1f} GB/s")
print(f"Ridge Point       : {RIDGE_POINT:.1f} FLOPs/Byte")

In [ ]:
def plot_roofline(points: dict, title: str = "TPU v6e (Trillium) Roofline (BF16)"):
    """points: {label: (arithmetic_intensity_flops_per_byte, achieved_tflops)}"""
    ai_x = np.logspace(-1, 4, 500)
    roof_y = np.minimum(PEAK_TFLOPS_BF16, (ai_x * PEAK_HBM_BW_GBS) / 1e3)

    plt.figure(figsize=(8, 5))
    plt.loglog(ai_x, roof_y, "k-", linewidth=2, label=f"{props.name} Ceiling ({PEAK_TFLOPS_BF16:.0f} TFLOPS, {PEAK_HBM_BW_GBS:.0f} GB/s)")
    plt.axvline(RIDGE_POINT, color="gray", linestyle="--", alpha=0.6, label=f"Ridge Point ({RIDGE_POINT:.0f} FLOPs/B)")

    for label, (ai, tflops) in points.items():
        plt.scatter([ai], [tflops], s=90, zorder=5, label=f"{label} ({tflops:.1f} TF/s)")
        plt.annotate(label, (ai, tflops), textcoords="offset points", xytext=(6, 6), fontsize=9)

    plt.xlabel("Arithmetic Intensity (FLOPs / Byte)")
    plt.ylabel("Performance (TFLOPS)")
    plt.title(title)
    plt.grid(True, which="both", linestyle=":", alpha=0.5)
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.show()

## 2. TorchTPU 4대 실행 모드 체계적 비교

TPU v6e(Trillium)는 MXU 크기가 **$256 \times 256$**으로 커지면서 Ridge Point가 **~560 FLOPs/Byte**에 달합니다.
- **Strict Eager**에서는 작은 연산들이 개별 실행 파일(`-O1`)로 쪼개져 **Microbubbles**와 과도한 HBM 왕복이 발생합니다.
- **DeferAndFuse**와 **`torch.compile(backend="tpu")`**는 연산들을 퓨전(`-O2`)하여 중간 활성화 텐서를 VMEM에 머물게 함으로써 연산 강도(Arithmetic Intensity)를 Ridge Point 우측으로 이동시킵니다.

In [ ]:
class RMSNormSwiGLUBlock(nn.Module):
    def __init__(self, hidden_dim: int = 2048, ffn_dim: int = 8192):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_dim))
        self.w_gate = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.w_up = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.w_down = nn.Linear(ffn_dim, hidden_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + 1e-6).to(x.dtype)
        h = x * rms * self.weight
        activated = F.silu(self.w_gate(h)) * self.w_up(h)
        return x + self.w_down(activated)

# 모든 주요 차원(tokens=8192, D=2048, FFN=8192)을 TPU v6e MXU(256x256) 배수로 정렬
B, S, D, FFN = 16, 512, 2048, 8192
tokens = B * S
model = RMSNormSwiGLUBlock(D, FFN).to(device=device, dtype=torch.bfloat16).eval()
x = torch.randn(B, S, D, device=device, dtype=torch.bfloat16).contiguous()

total_flops = 3 * 2 * tokens * D * FFN
fused_bytes = 2 * (tokens * D * 2 + 3 * D * FFN)
unfused_bytes = fused_bytes + 2 * (tokens * D * 6 + tokens * FFN * 5)

In [ ]:
def benchmark_fn(fn, x_in: torch.Tensor, warmup: int = 5, iters: int = 20) -> float:
    with torch.no_grad():
        for _ in range(warmup):
            _ = fn(x_in)
        torch.accelerator.synchronize()

        t0 = time.perf_counter()
        for _ in range(iters):
            out = fn(x_in)
        torch.accelerator.synchronize()
        elapsed = (time.perf_counter() - t0) / iters
    return elapsed

results = {}

# 1) Strict Eager (DEFER_NEVER)
with torch.tpu.set_eager_mode(torch.tpu.EagerMode.DEFER_NEVER):
    dt = benchmark_fn(model, x)
results["1. Strict Eager"] = (total_flops / unfused_bytes, (total_flops / dt) / 1e12, dt * 1e3)

# 2) Debug Eager (DEFER_NEVER_AND_LAUNCH_BLOCKING)
with torch.tpu.set_eager_mode(torch.tpu.EagerMode.DEFER_NEVER_AND_LAUNCH_BLOCKING):
    dt = benchmark_fn(model, x, warmup=2, iters=5)
results["2. Debug Eager"] = (total_flops / unfused_bytes, (total_flops / dt) / 1e12, dt * 1e3)

# 3) DeferAndFuse (Optimized Eager)
with torch.tpu.set_eager_mode(torch.tpu.EagerMode.DEFER_AND_FUSE):
    dt = benchmark_fn(model, x)
results["3. DeferAndFuse"] = (total_flops / ((fused_bytes + unfused_bytes) * 0.5), (total_flops / dt) / 1e12, dt * 1e3)

# 4) torch.compile(backend="tpu")
compiled_model = torch.compile(model, backend="tpu")
dt = benchmark_fn(compiled_model, x)
results["4. torch.compile"] = (total_flops / fused_bytes, (total_flops / dt) / 1e12, dt * 1e3)

print(f"{'Mode':<20} | {'Latency (ms)':>12} | {'Achieved TFLOPS':>15}")
print("-" * 54)
for name, (ai, tflops, ms) in results.items():
    print(f"{name:<20} | {ms:>12.3f} | {tflops:>15.2f}")

plot_roofline(
    {k: (v[0], v[1]) for k, v in results.items()},
    title="TorchTPU 4 Execution Modes on TPU v6e (Trillium) Roofline",
)

## 3. `torch.profiler` + `XProf` 트레이스 캡처

- 캡처 완료 후 CLI 터미널에서 `xprof --logdir=/tmp/torchtpu_xprof_modes --port=6006`을 띄우거나, Colab 환경이라면 아래 `%tensorboard` 셀을 실행해 확인합니다.

In [ ]:
from torch.tpu.profiler import TpuProfilerConfig

base_log_dir = pathlib.Path("/tmp/torchtpu_xprof_modes")

def capture_mode_profile(run_name: str, fn, eager_mode=None):
    out_dir = base_log_dir / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    tpu_cfg = TpuProfilerConfig(
        host_tracer_level=2,
        device_tracer_level=1,
        python_tracer_level=0,
        run_dir=out_dir,
    )
    ctx = torch.tpu.set_eager_mode(eager_mode) if eager_mode is not None else torch.no_grad()
    with ctx, torch.no_grad():
        with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.TPU,
            ],
            schedule=torch.profiler.schedule(wait=2, warmup=2, active=6, repeat=1),
            experimental_config=tpu_cfg,
            on_trace_ready=torch.profiler.tensorboard_trace_handler(str(out_dir)),
        ) as prof:
            for _ in range(10):
                _ = fn(x)
                torch.accelerator.synchronize()
                prof.step()
    print(f"Captured trace -> {out_dir}")

capture_mode_profile("strict_eager", model, eager_mode=torch.tpu.EagerMode.DEFER_NEVER)
capture_mode_profile("defer_and_fuse", model, eager_mode=torch.tpu.EagerMode.DEFER_AND_FUSE)
capture_mode_profile("torch_compile", compiled_model, eager_mode=None)
print("\n[CLI 뷰어 실행 명령어]: xprof --logdir=/tmp/torchtpu_xprof_modes --port=6006")

In [ ]:
# Google Colab UI에서 실행 중일 때만 주석을 해제해 인라인 TensorBoard(XProf)를 띄우세요.
# %load_ext tensorboard
# %tensorboard --logdir /tmp/torchtpu_xprof_modes